# Repository guide: Assisted transcription for manual review

Original code and recorded outputs retained. Read ../../docs/RUNNING.md before execution.


In [ ]:
# ============================================================
# CELL 1: Mount Drive and define project paths
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

QUEUE_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "transcription_queue_SPK001_SPK002.csv"
)

print("Project exists:", PROJECT_ROOT.exists())
print("Queue exists:", QUEUE_PATH.exists())

queue_df = pd.read_csv(QUEUE_PATH)

print("\nSegments:", len(queue_df))
print(
    "Speech minutes:",
    round(queue_df["duration_seconds"].sum() / 60, 2)
)

print("\nBy speaker:")
print(
    queue_df.groupby("speaker_group_id")
    .agg(
        segments=("segment_id", "count"),
        minutes=("duration_seconds", lambda x: round(x.sum()/60, 2))
    )
)

Mounted at /content/drive
Project exists: True
Queue exists: True

Segments: 1272
Speech minutes: 153.42

By speaker:
                  segments  minutes
speaker_group_id                   
SPK001                 702    94.84
SPK002                 570    58.58


In [ ]:
# ============================================================
# CELL 2: Verify all segment audio files
# ============================================================

missing_audio = []

for _, row in queue_df.iterrows():

    audio_path = PROJECT_ROOT / row["audio_path"]

    if not audio_path.exists():
        missing_audio.append({
            "segment_id": row["segment_id"],
            "audio_path": str(audio_path)
        })

print("Total segments:", len(queue_df))
print("Missing WAV files:", len(missing_audio))

if missing_audio:
    print("\nFirst missing files:")
    for x in missing_audio[:10]:
        print(x)
else:
    print("All audio files found ✓")

Total segments: 1272
Missing WAV files: 0
All audio files found ✓


In [ ]:
# ============================================================
# CELL 3: Install dependencies and check GPU
# ============================================================

!pip install -q transformers accelerate soundfile sentencepiece

import torch

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

Device: cuda
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
# ============================================================
# CELL 4: Load Tarifit MMS model
# ============================================================

from transformers import (
    AutoProcessor,
    AutoModelForCTC
)

MODEL_ID = "iukocha/mms-tachebdant-from-tarifit"

print("Loading:", MODEL_ID)

processor = AutoProcessor.from_pretrained(
    MODEL_ID
)

model = AutoModelForCTC.from_pretrained(
    MODEL_ID
)

model.to(DEVICE)
model.eval()

print("\nModel loaded successfully.")
print(
    "Parameters:",
    round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
    "M"
)

print("Vocabulary size:", model.config.vocab_size)
print("Pad / CTC blank:", model.config.pad_token_id)

Loading: iukocha/mms-tachebdant-from-tarifit


processor_config.json:   0%|          | 0.00/299 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/56.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.86GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]


Model loaded successfully.
Parameters: 964.7 M
Vocabulary size: 46
Pad / CTC blank: 42


In [ ]:
# ============================================================
# CELL 5: Normalize MMS draft output
# ============================================================

import re
import unicodedata

def normalize_mms_draft(text):

    if text is None:
        return ""

    text = unicodedata.normalize("NFC", str(text))
    text = text.lower()

    # Clear systematic MMS -> Corpus V1.1 differences
    text = text.replace("ə", "e")
    text = text.replace("š", "c")

    # Remove punctuation
    text = re.sub(r"[!?,.\-]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


print(normalize_mms_draft(
    "di lməɣrib, nəššin mamšira."
))

di lmeɣrib neccin mamcira


In [ ]:
# ============================================================
# CELL 6: Test MMS-assisted transcription on 5 SPK002 segments
# ============================================================

import numpy as np
import soundfile as sf
from tqdm.auto import tqdm

test_df = (
    queue_df[
        queue_df["speaker_group_id"] == "SPK002"
    ]
    .head(5)
    .copy()
)

test_results = []

for _, row in tqdm(
    test_df.iterrows(),
    total=len(test_df)
):

    audio_path = PROJECT_ROOT / row["audio_path"]

    audio, sr = sf.read(audio_path)

    inputs = processor(
        audio,
        sampling_rate=sr,
        return_tensors="pt"
    )

    input_values = inputs["input_values"].to(DEVICE)

    attention_mask = inputs.get("attention_mask")

    if attention_mask is not None:
        attention_mask = attention_mask.to(DEVICE)

    with torch.no_grad():
        logits = model(
            input_values=input_values,
            attention_mask=attention_mask
        ).logits

    predicted_ids = torch.argmax(
        logits,
        dim=-1
    )

    raw_text = processor.batch_decode(
        predicted_ids
    )[0]

    normalized_text = normalize_mms_draft(
        raw_text
    )

    test_results.append({
        "segment_id": row["segment_id"],
        "recording_id": row["recording_id"],
        "duration_seconds": row["duration_seconds"],
        "raw": raw_text,
        "normalized": normalized_text,
    })

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# CELL 7: Inspect test transcription drafts
# ============================================================

for x in test_results:

    print("=" * 70)
    print("SEGMENT :", x["segment_id"])
    print("RECORDING:", x["recording_id"])
    print("DURATION :", round(x["duration_seconds"], 2), "s")

    print("\nRAW MMS:")
    print(x["raw"])

    print("\nNORMALIZED DRAFT:")
    print(x["normalized"])

    print()

SEGMENT : REC034_SEG0001
RECORDING: REC034
DURATION : 2.61 s

RAW MMS:
zul aythma suthma ssitmɣa tirim mliḥ.

NORMALIZED DRAFT:
zul aythma suthma ssitmɣa tirim mliḥ

SEGMENT : REC034_SEG0002
RECORDING: REC034
DURATION : 12.69 s

RAW MMS:
akidhwumathum a yuba, nara təḥabəɣdzahwum d-gijen n təxrasth, zi thxrazin, n lpudkast nnəɣ dhaɣimit, amux dh zrim gifidyuya  nəssiwur x rwəqth, əmmi id ṭtawəqqaɛaš tuya.

NORMALIZED DRAFT:
akidhwumathum a yuba nara teḥabeɣdzahwum d gijen n texrasth zi thxrazin n lpudkast nneɣ dhaɣimit amux dh zrim gifidyuya nessiwur x rweqth emmi id ṭtaweqqaɛac tuya

SEGMENT : REC034_SEG0003
RECORDING: REC034
DURATION : 7.28 s

RAW MMS:
qbrim mrəbdha thaxrasta, awim d akidh xum rkas watayənxum niɣ rkasən qahwa, ḥma thiri dhaɣimita.

NORMALIZED DRAFT:
qbrim mrebdha thaxrasta awim d akidh xum rkas watayenxum niɣ rkasen qahwa ḥma thiri dhaɣimita

SEGMENT : REC034_SEG0004
RECORDING: REC034
DURATION : 8.72 s

RAW MMS:
tmizdhəkth taṣəbḥant, gətxərastaɣa zzak talbəɣ ad taris 

In [ ]:
# ============================================================
# CELL 8: Set transcription processing order
# ============================================================

speaker_order = {
    "SPK002": 0,
    "SPK001": 1,
}

queue_df["_speaker_order"] = (
    queue_df["speaker_group_id"]
    .map(speaker_order)
)

queue_df = (
    queue_df
    .sort_values([
        "_speaker_order",
        "recording_id",
        "start_seconds"
    ])
    .drop(columns="_speaker_order")
    .reset_index(drop=True)
)

print(
    queue_df.groupby(
        "speaker_group_id",
        sort=False
    ).size()
)

print("\nFirst recording:")
print(queue_df.iloc[0]["recording_id"])

print("\nLast recording:")
print(queue_df.iloc[-1]["recording_id"])

speaker_group_id
SPK002    570
SPK001    702
dtype: int64

First recording:
REC034

Last recording:
REC016


In [ ]:
# ============================================================
# CELL 9: Generate MMS drafts for all SPK001 + SPK002 segments
#         with checkpoint saving and resume support
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import soundfile as sf
import torch
from tqdm.auto import tqdm

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "transcription_queue_SPK001_SPK002_mms_drafts.csv"
)

# ------------------------------------------------------------
# Resume from previous saved progress if available
# ------------------------------------------------------------

if OUTPUT_PATH.exists():

    work_df = pd.read_csv(
        OUTPUT_PATH,
        keep_default_na=False
    )

    print("Resuming existing transcription file.")

else:

    work_df = queue_df.copy()

    for col in [
        "mms_draft_raw",
        "mms_draft_normalized",
        "final_transcription",
        "manual_review",
        "transcription_notes",
    ]:
        if col not in work_df.columns:
            work_df[col] = ""

    work_df["manual_review"] = (
        work_df["manual_review"]
        .replace("", "pending")
    )

    print("Starting new transcription batch.")


# ------------------------------------------------------------
# Determine which rows still need MMS drafts
# ------------------------------------------------------------

needs_draft = (
    work_df["mms_draft_raw"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

pending_indices = work_df.index[
    needs_draft
].tolist()

print("Total segments:", len(work_df))
print("Already drafted:", len(work_df) - len(pending_indices))
print("Remaining:", len(pending_indices))


# ------------------------------------------------------------
# Generate drafts
# ------------------------------------------------------------

SAVE_EVERY = 20

for count, idx in enumerate(
    tqdm(pending_indices),
    start=1
):

    row = work_df.loc[idx]

    audio_path = PROJECT_ROOT / row["audio_path"]

    try:

        audio, sr = sf.read(audio_path)

        inputs = processor(
            audio,
            sampling_rate=sr,
            return_tensors="pt"
        )

        input_values = (
            inputs["input_values"]
            .to(DEVICE)
        )

        attention_mask = inputs.get(
            "attention_mask"
        )

        if attention_mask is not None:
            attention_mask = (
                attention_mask.to(DEVICE)
            )

        with torch.no_grad():

            logits = model(
                input_values=input_values,
                attention_mask=attention_mask
            ).logits

        predicted_ids = torch.argmax(
            logits,
            dim=-1
        )

        raw_text = processor.batch_decode(
            predicted_ids
        )[0]

        normalized_text = normalize_mms_draft(
            raw_text
        )

        work_df.at[
            idx,
            "mms_draft_raw"
        ] = raw_text

        work_df.at[
            idx,
            "mms_draft_normalized"
        ] = normalized_text

    except Exception as e:

        work_df.at[
            idx,
            "transcription_notes"
        ] = f"MMS ERROR: {str(e)}"

        print(
            "\nError:",
            row["segment_id"],
            e
        )


    # --------------------------------------------------------
    # Periodic checkpoint
    # --------------------------------------------------------

    if count % SAVE_EVERY == 0:

        work_df.to_csv(
            OUTPUT_PATH,
            index=False,
            encoding="utf-8"
        )

        print(
            f"\nCheckpoint saved after "
            f"{count} new segments."
        )


# ------------------------------------------------------------
# Final save
# ------------------------------------------------------------

work_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print("\n===================================")
print("MMS draft generation finished.")
print("===================================")

print("Saved to:")
print(OUTPUT_PATH)

completed = (
    work_df["mms_draft_raw"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

print("\nDrafted:", completed)
print("Total:", len(work_df))

Starting new transcription batch.
Total segments: 1272
Already drafted: 0
Remaining: 1272


  0%|          | 0/1272 [00:00<?, ?it/s]

/tmp/ipykernel_1544/2731516907.py:135: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'zul aythma suthma ssitmɣa tirim mliḥ.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  work_df.at[
/tmp/ipykernel_1544/2731516907.py:140: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'zul aythma suthma ssitmɣa tirim mliḥ' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  work_df.at[



Checkpoint saved after 20 new segments.

Checkpoint saved after 40 new segments.

Checkpoint saved after 60 new segments.

Checkpoint saved after 80 new segments.

Checkpoint saved after 100 new segments.

Checkpoint saved after 120 new segments.

Checkpoint saved after 140 new segments.

Checkpoint saved after 160 new segments.

Checkpoint saved after 180 new segments.

Checkpoint saved after 200 new segments.

Checkpoint saved after 220 new segments.

Checkpoint saved after 240 new segments.

Checkpoint saved after 260 new segments.

Checkpoint saved after 280 new segments.

Checkpoint saved after 300 new segments.

Checkpoint saved after 320 new segments.

Checkpoint saved after 340 new segments.

Checkpoint saved after 360 new segments.

Checkpoint saved after 380 new segments.

Checkpoint saved after 400 new segments.

Checkpoint saved after 420 new segments.

Checkpoint saved after 440 new segments.

Checkpoint saved after 460 new segments.

Checkpoint saved after 480 new segmen

In [ ]:
# ============================================================
# CELL 10: Check MMS draft coverage
# ============================================================

draft_df = pd.read_csv(
    OUTPUT_PATH,
    keep_default_na=False
)

has_draft = (
    draft_df["mms_draft_raw"]
    .astype(str)
    .str.strip()
    .ne("")
)

print("Total segments:", len(draft_df))
print("MMS drafts:", has_draft.sum())
print("Missing drafts:", (~has_draft).sum())

print("\nBy speaker:")

for spk in ["SPK002", "SPK001"]:

    s = draft_df[
        draft_df["speaker_group_id"] == spk
    ]

    done = (
        s["mms_draft_raw"]
        .astype(str)
        .str.strip()
        .ne("")
        .sum()
    )

    print(
        f"{spk}: "
        f"{done}/{len(s)} drafts"
    )

Total segments: 1272
MMS drafts: 1272
Missing drafts: 0

By speaker:
SPK002: 570/570 drafts
SPK001: 702/702 drafts
